# Computing Eigenvectors and Eigenvalues

This notebook focuses on understanding and implementing the computation of eigenvectors and eigenvalues from first principles.

## Key Concepts
- **Eigenvalues**: Scalar values λ such that Av = λv
- **Eigenvectors**: Non-zero vectors v such that Av = λv where λ is an eigenvalue
- **Characteristic Equation**: det(A - λI) = 0, used to find eigenvalues
- **Characteristic Polynomial**: The polynomial obtained from det(A - λI)

## Imports and Setup

In [19]:
import numpy as np
import matplotlib.pyplot as plt

## Create Identity Matrix

Helper function to create identity matrices. This is a utility function used throughout the eigenvalue computation process whenever we need to construct (A - λI) matrices.
Uses `np.eye()` to create a matrix with ones on the diagonal and zeros elsewhere.

In [20]:
def create_identity_matrix(n):
    """
    Create an identity matrix of size n x n.
    
    Parameters:
    -----------
    n : int
        Dimension of the identity matrix
    
    Returns:
    --------
    I : ndarray
        Identity matrix of shape (n, n)
    """
    return np.eye(n)

## Compute Characteristic Polynomial

Computes the coefficients of the characteristic polynomial det(A - λI). 
The characteristic polynomial is a polynomial in λ whose roots are the eigenvalues of A.
Uses `np.poly()` to compute the polynomial coefficients and reverses them to get increasing order of powers.

In [21]:
def compute_characteristic_polynomial(A):
    """
    Compute the characteristic polynomial coefficients of matrix A.
    The characteristic polynomial is det(A - λI).
    
    Parameters:
    -----------
    A : ndarray
        Square matrix of shape (n, n)
    
    Returns:
    --------
    coefficients : ndarray
        Coefficients of the characteristic polynomial
        (in increasing order of powers)
    """
    n = A.shape[0]
    # Use numpy's built-in poly function which computes characteristic polynomial
    # poly returns coefficients in decreasing order of powers
    coefficients = np.poly(A)
    # Reverse to get increasing order
    return coefficients[::-1]

## Find Eigenvalues

Solves the characteristic equation det(A - λI) = 0 to find all eigenvalues.
Eigenvalues are the scalar values λ that satisfy the equation Av = λv.
Uses `np.linalg.eigvals()` which computes eigenvalues directly from the characteristic polynomial.

In [22]:
def find_eigenvalues(A):
    """
    Find eigenvalues by solving the characteristic equation det(A - λI) = 0.
    
    Parameters:
    -----------
    A : ndarray
        Square matrix of shape (n, n)
    
    Returns:
    --------
    eigenvalues : ndarray
        Array of eigenvalues
    """
    # np.linalg.eigvals computes eigenvalues from the characteristic polynomial
    return np.linalg.eigvals(A)

## Solve Homogeneous System

Solves the homogeneous system (A - λI)v = 0 to find eigenvectors.
For a given eigenvalue λ, we need to find all non-zero vectors v that satisfy this equation.
Uses Singular Value Decomposition (SVD) to find the null space - the eigenvector is the right singular vector corresponding to the smallest singular value.

In [23]:
def solve_homogeneous_system(A_minus_lambda_I):
    """
    Solve the homogeneous system (A - λI)v = 0 to find eigenvectors.
    
    Uses SVD to find the null space of the matrix.
    
    Parameters:
    -----------
    A_minus_lambda_I : ndarray
        Matrix (A - λI) of shape (n, n)
    
    Returns:
    --------
    eigenvector : ndarray
        An eigenvector corresponding to the homogeneous system
    """
    # Use SVD to find the null space
    U, S, Vt = np.linalg.svd(A_minus_lambda_I)
    # The eigenvector is the right singular vector corresponding to smallest singular value
    # (last row of V^T, or last column of V)
    eigenvector = Vt[-1, :]
    return eigenvector

## Find Eigenvector for Single Eigenvalue

Finds an eigenvector corresponding to a specific eigenvalue.
For each eigenvalue λ, we construct the matrix (A - λI) and solve (A - λI)v = 0 using the homogeneous system solver.
The result is normalized to have unit length for better conditioning and comparison.

In [24]:
def find_eigenvector_for_eigenvalue(A, eigenvalue, tolerance=1e-10):
    """
    Find an eigenvector for a given eigenvalue.
    Solves (A - λI)v = 0.
    
    Parameters:
    -----------
    A : ndarray
        Square matrix of shape (n, n)
    eigenvalue : float or complex
        The eigenvalue for which to find the eigenvector
    tolerance : float
        Tolerance for considering a value as effectively zero
    
    Returns:
    --------
    eigenvector : ndarray
        An eigenvector corresponding to the eigenvalue
    """
    n = A.shape[0]
    # Create (A - λI)
    A_minus_lambda_I = A - eigenvalue * create_identity_matrix(n)
    # Solve the homogeneous system
    eigenvector = solve_homogeneous_system(A_minus_lambda_I)
    return eigenvector

## Main Function: Compute Eigenvalues and Eigenvectors

In [25]:
def compute_eigensystem(A):
    """
    Compute all eigenvalues and corresponding eigenvectors of matrix A.
    
    Parameters:
    -----------
    A : ndarray
        Square matrix of shape (n, n)
    
    Returns:
    --------
    eigenvalues : ndarray
        Array of eigenvalues
    eigenvectors : ndarray
        Matrix where columns are eigenvectors.
        Column i corresponds to eigenvalues[i]
    """
    # Step 1: Find all eigenvalues
    eigenvalues = find_eigenvalues(A)
    
    # Step 2: For each eigenvalue, find corresponding eigenvector
    n = A.shape[0]
    eigenvectors = np.zeros((n, n), dtype=complex)
    
    for i, eigenvalue in enumerate(eigenvalues):
        eigenvector = find_eigenvector_for_eigenvalue(A, eigenvalue)
        # Normalize the eigenvector
        eigenvector = eigenvector / np.linalg.norm(eigenvector)
        eigenvectors[:, i] = eigenvector
    
    return eigenvalues, eigenvectors

## Example 1: 2x2 Matrix

In [26]:
# Define a simple 2x2 matrix
A_2x2 = np.array([
    [4, -2],
    [1, 1]
], dtype=float)

print("Matrix A:")
print(A_2x2)
print()

# Compute eigenvalues and eigenvectors
eigenvalues, eigenvectors = compute_eigensystem(A_2x2)

print("Eigenvalues:")
print(eigenvalues)
print()

print("Eigenvectors:")
print(eigenvectors)
print()

# Verify: Av = λv
print("Verification (Av = λv):")
for i, eigenvalue in enumerate(eigenvalues):
    v = eigenvectors[:, i]
    Av = A_2x2 @ v
    lambda_v = eigenvalue * v
    print(f"\nEigenvalue {i+1}: λ = {eigenvalue:.4f}")
    print(f"Av = {Av}")
    print(f"λv = {lambda_v}")
    print(f"Difference: {np.linalg.norm(Av - lambda_v):.2e}")

Matrix A:
[[ 4. -2.]
 [ 1.  1.]]

Eigenvalues:
[3. 2.]

Eigenvectors:
[[0.89442719+0.j 0.70710678+0.j]
 [0.4472136 +0.j 0.70710678+0.j]]

Verification (Av = λv):

Eigenvalue 1: λ = 3.0000
Av = [2.68328157+0.j 1.34164079+0.j]
λv = [2.68328157+0.j 1.34164079+0.j]
Difference: 4.97e-16

Eigenvalue 2: λ = 2.0000
Av = [1.41421356+0.j 1.41421356+0.j]
λv = [1.41421356+0.j 1.41421356+0.j]
Difference: 0.00e+00


## Example 2: 3x3 Symmetric Matrix

In [27]:
# Define a 3x3 symmetric matrix
A_3x3 = np.array([
    [2, 1, 0],
    [1, 2, 1],
    [0, 1, 2]
], dtype=float)

print("Matrix A:")
print(A_3x3)
print()

# Compute eigenvalues and eigenvectors
eigenvalues, eigenvectors = compute_eigensystem(A_3x3)

print("Eigenvalues:")
print(eigenvalues)
print()

print("Eigenvectors (as columns):")
print(eigenvectors)
print()

# Verify for first eigenvalue
print("Verification (Av = λv) for first eigenvalue:")
v = eigenvectors[:, 0]
Av = A_3x3 @ v
lambda_v = eigenvalues[0] * v
print(f"Av = {Av}")
print(f"λv = {lambda_v}")
print(f"Error: {np.linalg.norm(Av - lambda_v):.2e}")

Matrix A:
[[2. 1. 0.]
 [1. 2. 1.]
 [0. 1. 2.]]

Eigenvalues:
[3.41421356 2.         0.58578644]

Eigenvectors (as columns):
[[ 0.5       +0.j -0.70710678+0.j  0.5       +0.j]
 [ 0.70710678+0.j  0.        +0.j -0.70710678+0.j]
 [ 0.5       +0.j  0.70710678+0.j  0.5       +0.j]]

Verification (Av = λv) for first eigenvalue:
Av = [1.70710678+0.j 2.41421356+0.j 1.70710678+0.j]
λv = [1.70710678+0.j 2.41421356+0.j 1.70710678+0.j]
Error: 3.50e-15


## Comparison with NumPy's Built-in Function

In [28]:
# Compare with numpy's linalg.eig
print("Using numpy.linalg.eig:")
np_eigenvalues, np_eigenvectors = np.linalg.eig(A_3x3)

print("NumPy Eigenvalues:")
print(np_eigenvalues)
print()

print("NumPy Eigenvectors (as columns):")
print(np_eigenvectors)
print()

print("\nOur Implementation Eigenvalues:")
print(eigenvalues)
print()

print("Difference in eigenvalues:")
print(np.linalg.norm(np_eigenvalues - eigenvalues))

Using numpy.linalg.eig:
NumPy Eigenvalues:
[3.41421356 2.         0.58578644]

NumPy Eigenvectors (as columns):
[[-5.00000000e-01  7.07106781e-01  5.00000000e-01]
 [-7.07106781e-01  4.05405432e-16 -7.07106781e-01]
 [-5.00000000e-01 -7.07106781e-01  5.00000000e-01]]


Our Implementation Eigenvalues:
[3.41421356 2.         0.58578644]

Difference in eigenvalues:
0.0


## Summary

We've implemented the eigenvalue decomposition from first principles:

1. **create_identity_matrix()**: Helper to create I matrices
2. **compute_characteristic_polynomial()**: Computes det(A - λI) coefficients
3. **find_eigenvalues()**: Solves the characteristic equation to find λ values
4. **solve_homogeneous_system()**: Solves (A - λI)v = 0 using SVD
5. **find_eigenvector_for_eigenvalue()**: Finds eigenvector for a specific eigenvalue
6. **compute_eigensystem()**: Main function combining all steps

This modular approach makes it easy to understand each step of the eigendecomposition process.